### Expected performance estimation - Absolute and average estimation error tables

In [2]:
import os
import re
import glob
import pandas as pd

# ---------- Config / mappings ----------
STRATEGY_LABELS = {
    "naive":     "Uncal",
    "isotonic":  "Iso",
    "sigmoid":   "Platt",
    "regva":        "RegVA",
    "avgva":       "AvgVA",
}
STRATEGY_ORDER = ["Uncal", "Iso", "Platt", "RegVA", "AvgVA"]

ALGO_LABELS = {"tree": "DT", "rf": "RF", "xgb": "XGB"}
ALGO_ORDER  = ["DT", "RF", "XGB"]

# dataset_strategy_metric_algorithm.csv
FNAME_RE = re.compile(
    r"^(?:XXX_)?(?P<dataset>[^_]+)_(?P<strategy>[^_]+)_(?P<algorithm>[^_]+)\.csv$",
    re.IGNORECASE
)

def parse_filename_safe(path):
    base = os.path.basename(path)
    m = FNAME_RE.match(base)
    if not m:
        return None
    d = m.groupdict()
    return d["dataset"], d["strategy"].lower(), d["algorithm"].lower()

def load_results_df(datasets, folder_path="."):
    """
    Loads all CSVs for the given datasets and computes:
      - abs_error   = |estimate - accuracy|
      - signed_error=  estimate - accuracy
    Returns columns include: dataset, rejection_rate, strategy, algorithm, abs_error, signed_error
    """
    dfs, skipped = [], []

    for file in glob.glob(os.path.join(folder_path, "*.csv")):
        parsed = parse_filename_safe(file)
        if not parsed:
            skipped.append((file, "filename pattern mismatch"))
            continue

        dataset, strategy, algorithm = parsed
        if dataset not in datasets:
            continue

        if algorithm not in ALGO_LABELS:
            skipped.append((file, f"unknown algorithm: {algorithm}"))
            continue
        if strategy not in STRATEGY_LABELS:
            skipped.append((file, f"unknown strategy: {strategy}"))
            continue

        try:
            df = pd.read_csv(file)
        except Exception as e:
            skipped.append((file, f"read error: {e}"))
            continue

        required = {"rejection_rate", "estimate", "accuracy"}
        if not required.issubset(df.columns):
            skipped.append((file, f"missing columns; found {list(df.columns)}"))
            continue

        df = df.copy()
        df["signed_error"] = (df["estimate"] - df["accuracy"])
        df["abs_error"]    = df["signed_error"].abs()

        df["dataset"]   = dataset
        df["strategy"]  = strategy
        df["algorithm"] = algorithm
        dfs.append(df)

    if not dfs:
        msg = "No valid files found for the requested datasets."
        if skipped:
            msg += " Skipped: " + "; ".join([f"{os.path.basename(f)} → {r}" for f, r in skipped])
        raise RuntimeError(msg)

    return pd.concat(dfs, ignore_index=True)

def _strip_leading_zero_str(s: str) -> str:
    """Turn '0.123' -> '.123' and '-0.123' -> '-.123'."""
    if s.startswith("0."):
        return s[1:]
    if s.startswith("-0."):
        return "-" + s[2:]
    return s

def make_tables_by_algorithm_latex(
    results_df,
    caption_prefix="Accuracy estimation error for each rejection rate, averaged across all datasets.",
    label_prefix="tab:error_by_rejection",
    float_fmt="{:.3f}",
    fill="."
):
    """
    Returns a dict: { 'DT': latex_str, 'RF': latex_str, 'XGB': latex_str }
    Each table:
      - rows: rejection_rate 0.1..0.9 (shown as 10%..90%) + Mean
      - columns: for each strategy, two columns (Abs, Signed)
      - values: mean across datasets for that algorithm/strategy/rejection_rate
    """
    # Average across datasets: rejection_rate × algorithm × strategy
    grouped = (
        results_df
        .groupby(["rejection_rate", "algorithm", "strategy"], as_index=False)
        .agg(abs_error=("abs_error", "mean"), signed_error=("signed_error", "mean"))
    )

    grouped["Algorithm"] = grouped["algorithm"].map(ALGO_LABELS)
    grouped["Strategy"]  = grouped["strategy"].map(STRATEGY_LABELS)
    grouped = grouped.dropna(subset=["Algorithm", "Strategy"])

    # Standard row order (true rejection rates)
    rej_order = [round(x / 10, 1) for x in range(1, 10)]

    def _fmt(x):
        if pd.isnull(x):
            return fill
        s = float_fmt.format(float(x))
        return _strip_leading_zero_str(s)

    tables = {}

    for alg_disp in ALGO_ORDER:
        g_alg = grouped[grouped["Algorithm"] == alg_disp].copy()

        # Pivot to MultiIndex columns: (Strategy, Metric)
        pivot = g_alg.pivot_table(
            index="rejection_rate",
            columns=["Strategy"],
            values=["abs_error", "signed_error"],
            aggfunc="first"
        )

        pivot = pivot.swaplevel(0, 1, axis=1).sort_index(axis=1)

        # Reindex rows and expected columns
        pivot.index = pivot.index.round(1)
        full_cols = pd.MultiIndex.from_product([STRATEGY_ORDER, ["abs_error", "signed_error"]])
        pivot = pivot.reindex(index=rej_order, columns=full_cols)

        # Column means
        col_means = pivot.mean(axis=0, skipna=True)

        # Format cells
        pivot_fmt = pivot.applymap(_fmt)

        # --- Build LaTeX table ---
        # 1st col + 2 per strategy
        ncols = 1 + 2 * len(STRATEGY_ORDER)
        tab_spec = "c" * ncols

        # Header lines
        # Strategy grouped headers:
        #   & \multicolumn{2}{c}{Uncal} & ... \\
        strat_group = " & ".join([f"\\multicolumn{{2}}{{c}}{{{s}}}" for s in STRATEGY_ORDER])
        subheader   = " & ".join(["Abs & Signed"] * len(STRATEGY_ORDER))

        lines = []
        lines.append("\\begin{table}[!ht]")
        lines.append("    \\centering")
        lines.append(f"    \\begin{{tabular}}{{{tab_spec}}}")
        lines.append("    \\toprule")
        lines.append(f"    & \\multicolumn{{{2*len(STRATEGY_ORDER)}}}{{c}}{{{alg_disp}}} \\\\")
        lines.append(f"    Rejection rate & {strat_group} \\\\")
        lines.append(f"    & {subheader} \\\\")
        lines.append("    \\midrule")

        # Body rows
        for r in rej_order:
            pct = int(round(r * 100))
            row = [f"{pct}\\%"]
            for strat in STRATEGY_ORDER:
                row.append(pivot_fmt.loc[r, (strat, "abs_error")])
                row.append(pivot_fmt.loc[r, (strat, "signed_error")])
            lines.append("    " + " & ".join(row) + " \\\\")

        # Mean row
        lines.append("    \\midrule")
        mean_cells = []
        for strat in STRATEGY_ORDER:
            mean_cells.append(_fmt(col_means.get((strat, "abs_error"))))
            mean_cells.append(_fmt(col_means.get((strat, "signed_error"))))
        lines.append("    Mean & " + " & ".join(mean_cells) + " \\\\")

        lines.append("    \\bottomrule")
        lines.append("    \\end{tabular}")

        caption = f"{caption_prefix} ({alg_disp})."
        label   = f"{label_prefix}:{alg_disp.lower()}"
        lines.append(f"    \\caption{{{caption}}}")
        lines.append(f"    \\label{{{label}}}")
        lines.append("\\end{table}")

        tables[alg_disp] = "\n".join(lines)

    return tables

# ---------- Example usage ----------
if __name__ == "__main__":
    datasets_to_process = [
        "adult","bank8FM","phishing","puma8NH","phoneme","krkp","german",
        "diabetes","kc2","kc3","heartC","heartH","heartS","hepati", "pc4", "deltae",
        "elevators", "raising", "liver", "magic", "vertebral", "qsar", "eeg", "cpu"
    ]

    folder = "./exp_performance_results"
    df = load_results_df(datasets_to_process, folder)

    tables = make_tables_by_algorithm_latex(
        df,
        caption_prefix="Accuracy estimation error for each rejection rate, averaged across all datasets.",
        label_prefix="tab:error_by_rejection",
        float_fmt="{:.3f}",
        fill="."
    )

    # Print all tables (DT, RF, XGB)
    for alg in ALGO_ORDER:
        print(tables[alg])
        print("\n")

\begin{table}[!ht]
    \centering
    \begin{tabular}{ccccccccccc}
    \toprule
    & \multicolumn{10}{c}{DT} \\
    Rejection rate & \multicolumn{2}{c}{Uncal} & \multicolumn{2}{c}{Iso} & \multicolumn{2}{c}{Platt} & \multicolumn{2}{c}{RegVA} & \multicolumn{2}{c}{AvgVA} \\
    & Abs & Signed & Abs & Signed & Abs & Signed & Abs & Signed & Abs & Signed \\
    \midrule
    10\% & .036 & .036 & .010 & .010 & .008 & .006 & .010 & -.009 & .003 & .001 \\
    20\% & .038 & .038 & .011 & .011 & .006 & .001 & .011 & -.011 & .003 & .001 \\
    30\% & .039 & .039 & .011 & .011 & .009 & -.005 & .011 & -.011 & .003 & .001 \\
    40\% & .042 & .042 & .011 & .011 & .013 & -.011 & .010 & -.010 & .004 & .001 \\
    50\% & .041 & .041 & .010 & .010 & .018 & -.018 & .009 & -.009 & .003 & .001 \\
    60\% & .040 & .040 & .012 & .012 & .024 & -.024 & .009 & -.009 & .005 & .001 \\
    70\% & .040 & .040 & .012 & .012 & .028 & -.028 & .010 & -.009 & .005 & .000 \\
    80\% & .040 & .040 & .012 & .012 & .032 & 

/tmp/ipykernel_221691/3786940494.py:153: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(_fmt)
/tmp/ipykernel_221691/3786940494.py:153: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(_fmt)
/tmp/ipykernel_221691/3786940494.py:153: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(_fmt)


### Worst-case performance estimation - Absolute and average estimation error tables

In [ ]:
import os
import re
import glob
import pandas as pd

# ---------- Config / mappings ----------
STRATEGY_LABELS = {
    "naive":     "Uncal",
    "isotonic":  "Iso",
    "sigmoid":   "Platt",
    "regva":        "RegVA",
    "avgva":       "AvgVA",
}
STRATEGY_ORDER = ["Uncal", "Iso", "Platt", "RegVA", "AvgVA"]

ALGO_LABELS = {"rf": "RF"}
ALGO_ORDER  = ["RF"]

# ✅ Regex allows metric section to be optional
FNAME_RE = re.compile(
    r"^(?:XXX_)?(?P<dataset>[^_]+)_(?P<strategy>[^_]+)(?:_(?P<metric>[^_]+))?_(?P<algorithm>[^_]+)\.csv$",
    re.IGNORECASE
)

def parse_filename_safe(path):
    base = os.path.basename(path)
    m = FNAME_RE.match(base)
    if not m:
        return None
    d = m.groupdict()
    return (
        d["dataset"],
        d["strategy"].lower(),
        (d["metric"] or "").lower(),
        d["algorithm"].lower(),
    )

def load_results_df(folder_path="."):
    dfs, skipped = [], []

    for file in glob.glob(os.path.join(folder_path, "*.csv")):
        parsed = parse_filename_safe(file)
        if not parsed:
            skipped.append((file, "filename pattern mismatch"))
            continue

        dataset, strategy, metric, algorithm = parsed
        if algorithm not in ALGO_LABELS:
            skipped.append((file, f"unknown algorithm: {algorithm}"))
            continue
        if strategy not in STRATEGY_LABELS:
            skipped.append((file, f"unknown strategy: {strategy}"))
            continue

        try:
            df = pd.read_csv(file)
        except Exception as e:
            skipped.append((file, f"read error: {e}"))
            continue

        required = {"rejection_rate", "abs_error", "signed_error"}
        if not required.issubset(df.columns):
            skipped.append((file, f"missing columns; found {list(df.columns)}"))
            continue

        df = df.copy()
        df["rejection_rate"] = df["rejection_rate"].astype(float).round(1)
        df["dataset"]  = dataset
        df["strategy"] = strategy
        df["algorithm"]= algorithm
        dfs.append(df)

    if not dfs:
        msg = "❌ No valid files found."
        if skipped:
            msg += "\nSkipped files:\n" + "\n".join([f"- {os.path.basename(f)} → {r}" for f, r in skipped])
        raise RuntimeError(msg)

    print(f"✅ Loaded {len(dfs)} files successfully.")
    if skipped:
        print("⚠️ Skipped files:")
        for f, reason in skipped:
            print(f"  - {os.path.basename(f)} → {reason}")

    return pd.concat(dfs, ignore_index=True)

def _fmt_value(x, float_fmt="{:.3f}", fill="."):
    """Format floats like .123 instead of 0.123."""
    if pd.isnull(x):
        return fill
    s = float_fmt.format(float(x))
    if s.startswith("0."):
        s = s[1:]
    if s.startswith("-0."):
        s = "-" + s[2:]
    return s

def make_combined_table_latex(results_df,
                              caption="Average absolute and signed error per rejection rate",
                              label="tab:abs_signed_combined",
                              float_fmt="{:.3f}",
                              fill="."):
    """
    Builds a LaTeX table combining both absolute and signed errors.
    Columns alternate: Uncal(abs), Uncal(signed), Iso(abs), Iso(signed), ...
    """

    results_df["rejection_rate"] = results_df["rejection_rate"].astype(float).round(1)

    # ---- Aggregate both error types ----
    abs_grouped = (
        results_df
        .groupby(["rejection_rate", "strategy"], as_index=False)["abs_error"]
        .mean()
        .rename(columns={"abs_error": "value"})
    )
    abs_grouped["ErrorType"] = "abs"

    signed_grouped = (
        results_df
        .groupby(["rejection_rate", "strategy"], as_index=False)["signed_error"]
        .mean()
        .rename(columns={"signed_error": "value"})
    )
    signed_grouped["ErrorType"] = "signed"

    combined = pd.concat([abs_grouped, signed_grouped], ignore_index=True)
    combined["Strategy"] = combined["strategy"].map(STRATEGY_LABELS)

    # ---- Pivot to create alternating columns ----
    pivot = combined.pivot_table(
        index="rejection_rate",
        columns=["Strategy", "ErrorType"],
        values="value",
        aggfunc="mean"
    )

    # Ensure correct order
    rej_order = [round(x / 10, 1) for x in range(1, 10)]
    col_order = []
    for s in STRATEGY_ORDER:
        col_order.append((s, "abs"))
        col_order.append((s, "signed"))

    pivot = pivot.reindex(index=rej_order, columns=pd.MultiIndex.from_tuples(col_order))
    col_means = pivot.mean(axis=0, skipna=True)

    # Format for LaTeX
    pivot_fmt = pivot.applymap(lambda x: _fmt_value(x, float_fmt, fill))

    # ---- Build LaTeX ----
    lines = []
    lines.append("\\begin{table}[!ht]")
    lines.append("    \\centering")

    # Header line 1
    header_main = "         Rejection rate"
    for s in STRATEGY_ORDER:
        header_main += f" & \\multicolumn{{2}}{{c}}{{{s}}}"
    lines.append("    \\begin{tabular}{" + "l" + "cc"*len(STRATEGY_ORDER) + "}")
    lines.append("    \\toprule")
    lines.append(header_main + " \\\\")
    # Header line 2
    header_sub = "         "
    for _ in STRATEGY_ORDER:
        header_sub += " & abs & signed"
    lines.append(header_sub + " \\\\")
    lines.append("    \\midrule")

    # Body
    for r in rej_order:
        pct = int(round(r * 100))
        row = [f"{pct}\\%"]
        for s in STRATEGY_ORDER:
            row.append(pivot_fmt.loc[r, (s, "abs")])
            row.append(pivot_fmt.loc[r, (s, "signed")])
        lines.append("    " + " & ".join(row) + " \\\\")

    # Mean row
    lines.append("    \\midrule")
    row = ["Mean"]
    for s in STRATEGY_ORDER:
        row.append(_fmt_value(col_means.get((s, "abs")), float_fmt, fill))
        row.append(_fmt_value(col_means.get((s, "signed")), float_fmt, fill))
    lines.append("    " + " & ".join(row) + " \\\\")

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")
    lines.append(f"    \\caption{{{caption}}}")
    lines.append(f"    \\label{{{label}}}")
    lines.append("\\end{table}")

    return "\n".join(lines)

# ---------- Utility ----------
def discover_datasets(folder_path="."):
    datasets = set()
    for file in glob.glob(os.path.join(folder_path, "*.csv")):
        parsed = parse_filename_safe(file)
        if parsed:
            dataset, strategy, metric, algorithm = parsed
            datasets.add(dataset)
    return sorted(datasets)




# ---------- Main ----------
if __name__ == "__main__":
    folder = "./lower_bound_results"

    datasets_to_process = discover_datasets(folder)
    print(f"📂 Found {len(datasets_to_process)} datasets: {', '.join(datasets_to_process)}")

    df = load_results_df(folder)

    print(f"\n✅ Processed {df['dataset'].nunique()} unique datasets "
          f"from {len(glob.glob(os.path.join(folder, '*.csv')))} CSV files, totaling {len(df)} rows.\n")

    latex_str = make_combined_table_latex(
        df,
        caption="Combined absolute and signed estimation errors per rejection rate (averaged across all datasets).",
        label="tab:abs_signed_combined",
        float_fmt="{:.3f}",
        fill="."
    )

    print("\nGenerated Combined LaTeX Table:")
    print("-------------------------------")
    print(latex_str)
    



### Expected performance estimation - CDs

In [ ]:
import numpy as np
import pandas as pd
from critdd import Diagram

# Reuse your mappings/orders
# STRATEGY_LABELS, STRATEGY_ORDER, ALGO_LABELS, ALGO_ORDER

def cd_tikz_for_algo_rejrate(
    results_df: pd.DataFrame,
    *,
    algorithm: str,
    rejection_rate: float,
    alpha: float = 0.05,
    adjustment: str = "holm",      # or "bonferroni"
    reverse_x: bool = True,
    as_document: bool = False,
    axis_title: str,
) -> str:
    """
    Build a critdd CD diagram (one) for a given algorithm and rejection_rate,
    using per-dataset mean absolute error as the outcome (minimize).
    Returns TikZ code as a string.

    Parameters
    ----------
    algorithm:
        Either raw key ("tree","rf","xgb") OR display label ("DT","RF","XGB").
    rejection_rate:
        Float in [0,1], e.g. 0.1, 0.2, ... matching results_df["rejection_rate"].
        (If your CSV column is actually coverage, pass rejection_rate=(1-coverage)
         or precompute a 'rej_rate_true' column and swap below.)
    """

    # ---- Normalize algorithm input to raw key used in results_df ----
    algo_key = algorithm.lower()
    if algo_key in ALGO_LABELS:  # raw key
        pass
    else:
        # maybe user passed "DT"/"RF"/"XGB"
        inv = {v.lower(): k for k, v in ALGO_LABELS.items()}
        if algo_key not in inv:
            raise ValueError(f"Unknown algorithm '{algorithm}'. "
                             f"Use one of {list(ALGO_LABELS.keys())} or {list(ALGO_LABELS.values())}.")
        algo_key = inv[algo_key]

    # ---- Filter and compute mean error per dataset x strategy ----
    df = results_df.copy()

    # If your column is actually COVERAGE, uncomment this and use 'rej_rate_true' below:
    # df["rej_rate_true"] = 1.0 - df["rejection_rate"]

    df = df[(df["algorithm"] == algo_key)].copy()

    # pick the rejection-rate column you want to slice on
    rr_col = "rejection_rate"      # or "rej_rate_true" if you created it above

    # robust float match (round to 1 decimal like in your table)
    df[rr_col] = df[rr_col].astype(float).round(1)
    rr = round(float(rejection_rate), 1)
    df = df[df[rr_col] == rr].copy()

    if df.empty:
        raise ValueError(f"No rows found for algorithm={algorithm} and {rr_col}={rr}")

    # mean error within dataset x strategy (in case multiple rows per file / seeds)
    g = (
        df.groupby(["dataset", "strategy"], as_index=False)["abs_error"]
          .mean()
    )

    # map strategy keys to display labels and pivot
    g["Strategy"] = g["strategy"].map(STRATEGY_LABELS)
    g = g.dropna(subset=["Strategy"])

    mat = g.pivot(index="dataset", columns="Strategy", values="abs_error")

    # enforce fixed strategy order; critdd expects a fixed treatment order
    mat = mat.reindex(columns=STRATEGY_ORDER)

    # critdd requires every observation (dataset) has outcomes for every treatment
    mat = mat.dropna(axis=0, how="any")

    if mat.shape[0] < 2:
        raise ValueError(
            f"After dropping missing strategies, only {mat.shape[0]} dataset(s) remain. "
            "CD diagrams need multiple observations."
        )

    # ---- Build CD diagram (minimization => maximize_outcome=False) ----
    diagram = Diagram(
        mat.to_numpy(),
        treatment_names=list(mat.columns),
        maximize_outcome=False,
    )  # Diagram usage per docs :contentReference[oaicite:1]{index=1}

    # ---- Export to TikZ string ----
    axis_opts = {}
    if axis_title is None:
        axis_title = f"{ALGO_LABELS[algo_key]} @ r={rr:.1f}"
    axis_opts["title"] = axis_title

    tikz = diagram.to_str(
        alpha=alpha,
        adjustment=adjustment,
        reverse_x=reverse_x,
        as_document=as_document,
        axis_options=axis_opts,
    )  

    return tikz


# ---------------- Example usage ----------------
if __name__ == "__main__":
    tikz_code = cd_tikz_for_algo_rejrate(
        df,
        algorithm="XGB",          # or "rf"
        rejection_rate=0.9,
        alpha=0.05,
        adjustment="holm",
        reverse_x=True,
        as_document=False,
        axis_title="Subtitle" # set True if you want a standalone compilable .tex
    )
    print(tikz_code)



\begin{tikzpicture}[
  treatment line/.style={rounded corners=1.5pt, line cap=round, shorten >=1pt},
  treatment label/.style={font=\small},
  group line/.style={ultra thick},
]

\begin{axis}[
  clip={false},
  axis x line={center},
  axis y line={none},
  axis line style={-},
  xmin={1},
  ymax={0},
  scale only axis={true},
  width={\axisdefaultwidth},
  ticklabel style={anchor=south, yshift=1.3*\pgfkeysvalueof{/pgfplots/major tick length}, font=\small},
  every tick/.style={draw=black},
  major tick style={yshift=.5*\pgfkeysvalueof{/pgfplots/major tick length}},
  minor tick style={yshift=.5*\pgfkeysvalueof{/pgfplots/minor tick length}},
  title style={yshift=\baselineskip},
  xmax={5},
  ymin={-3.5},
  height={4\baselineskip},
  xtick={1,2,3,4,5},
  minor x tick num={3},
  x dir={reverse},
  title={Subtitle},
]

\draw[treatment line] ([yshift=-2pt] axis cs:1.6666666666666667, 0) |- (axis cs:1.25, -2.5)
  node[treatment label, anchor=west] {AvgVA};
\draw[treatment line] ([yshift=-2p

### Expected performance estimation - Per-dataset absolute errors

In [12]:
import os
import re
import glob
import pandas as pd

# ---------- Config / mappings ----------
STRATEGY_LABELS = {
    "naive":     "Uncal",
    "isotonic":  "Iso",
    "sigmoid":   "Platt",
    "regva":        "RegVA",
    "avgva":       "AvgVA",
}
STRATEGY_ORDER = ["Uncal", "Iso", "Platt", "RegVA", "AvgVA"]

ALGO_LABELS = {"tree": "DT", "rf": "RF", "xgb": "XGB"}
ALGO_ORDER  = ["DT", "RF", "XGB"]

# dataset_strategy_metric_algorithm.csv
FNAME_RE = re.compile(
    r"^(?:XXX_)?(?P<dataset>[^_]+)_(?P<strategy>[^_]+)_(?P<algorithm>[^_]+)\.csv$",
    re.IGNORECASE
)

def parse_filename_safe(path):
    base = os.path.basename(path)
    m = FNAME_RE.match(base)
    if not m:
        return None
    d = m.groupdict()
    return d["dataset"], d["strategy"].lower(), d["algorithm"].lower()

def load_results_df(datasets, folder_path="."):
    """
    Loads all CSVs for the given datasets.
    IMPORTANT: CSV column 'rejection_rate' is actually COVERAGE in [0,1]. (As per your original note.)
    Returns columns: dataset, strategy, algorithm, accuracy, estimate, rejection_rate, ...
    """
    dfs, skipped = [], []

    for file in glob.glob(os.path.join(folder_path, "*.csv")):
        parsed = parse_filename_safe(file)
        if not parsed:
            skipped.append((file, "filename pattern mismatch"))
            continue

        dataset, strategy, algorithm = parsed
        if dataset not in datasets:
            continue

        if algorithm not in ALGO_LABELS:
            skipped.append((file, f"unknown algorithm: {algorithm}"))
            continue
        if strategy not in STRATEGY_LABELS:
            skipped.append((file, f"unknown strategy: {strategy}"))
            continue

        try:
            df = pd.read_csv(file)
        except Exception as e:
            skipped.append((file, f"read error: {e}"))
            continue

        required = {"rejection_rate", "estimate", "accuracy"}
        if not required.issubset(df.columns):
            skipped.append((file, f"missing columns; found {list(df.columns)}"))
            continue

        df = df.copy()
        df["dataset"]   = dataset
        df["strategy"]  = strategy
        df["algorithm"] = algorithm
        dfs.append(df)

    if not dfs:
        msg = "No valid files found for the requested datasets."
        if skipped:
            msg += " Skipped: " + "; ".join([f"{os.path.basename(f)} → {r}" for f, r in skipped])
        raise RuntimeError(msg)

    return pd.concat(dfs, ignore_index=True)

def _strip_leading_zero_str(s: str) -> str:
    if s.startswith("0."):
        return s[1:]
    if s.startswith("-0."):
        return "-" + s[2:]
    return s

def _latex_escape(s: str) -> str:
    return (s.replace("\\", "\\textbackslash{}")
             .replace("_", "\\_")
             .replace("%", "\\%")
             .replace("&", "\\&")
             .replace("#", "\\#")
             .replace("{", "\\{")
             .replace("}", "\\}")
             .replace("$", "\\$")
             .replace("~", "\\textasciitilde{}")
             .replace("^", "\\textasciicircum{}"))

def make_dataset_err_table_all_algos_latex(
    results_df: pd.DataFrame,
    target_rejection_rate: float,
    rr_tol: float = 1e-6,
    algorithms=ALGO_ORDER,
    strategies=STRATEGY_ORDER,
    dataset_order = None,  # if None, use sorted order of datasets in the filtered df
    caption_prefix="Absolute estimation error",
    label_prefix="tab:err_all_algos",
    float_fmt="{:.3f}",
    fill="."
):
    """
    Single LaTeX table (all algorithms together):
      Rows: datasets
      Cols: for each Algorithm -> for each Strategy -> Err only
      Plus two summary rows:
        - Mean: mean Err across datasets per (Algorithm, Strategy)
        - Mean rank (Err): for each Algorithm separately, rank strategies per dataset by Err (lower better),
                           average ranks across datasets. (Placed under that Algorithm block.)

    Selection: rejection_rate ~= target_rejection_rate within rr_tol.
    Aggregation: mean across repeats/folds for that rejection rate.
    Returns: latex_str
    """
    df = results_df.copy()
    df["rejection_rate"] = df["rejection_rate"].astype(float)

    sel = df[(df["rejection_rate"] - float(target_rejection_rate)).abs() <= float(rr_tol)].copy()
    if sel.empty:
        avail = sorted(df["rejection_rate"].round(6).dropna().unique())
        raise RuntimeError(
            f"No rows found for target_rejection_rate={target_rejection_rate} within rr_tol={rr_tol}. "
            f"Available rejection_rate values (sample): {avail[:20]}{' ...' if len(avail) > 20 else ''}"
        )

    sel["Algorithm"] = sel["algorithm"].map(ALGO_LABELS)
    sel["Strategy"]  = sel["strategy"].map(STRATEGY_LABELS)
    sel = sel.dropna(subset=["Algorithm", "Strategy"])

    def _fmt(x):
        if pd.isnull(x):
            return fill
        return _strip_leading_zero_str(float_fmt.format(float(x)))

    # mean Acc/Est per dataset x algorithm x strategy, then Err
    acc = (sel.groupby(["dataset", "Algorithm", "Strategy"], as_index=False)["accuracy"].mean()
             .rename(columns={"accuracy": "Acc"}))
    est = (sel.groupby(["dataset", "Algorithm", "Strategy"], as_index=False)["estimate"].mean()
             .rename(columns={"estimate": "Est"}))

    merged = acc.merge(est, on=["dataset", "Algorithm", "Strategy"], how="outer")
    merged["Err"] = (merged["Est"] - merged["Acc"]).abs()

    # Pivot: rows=datasets, cols=(Algorithm, Strategy) values=Err
    pivot = merged.pivot(index="dataset", columns=["Algorithm", "Strategy"], values="Err")

    if dataset_order is not None:
        pivot = pivot.reindex(dataset_order)

    # enforce column order
    desired_cols = [(a, s) for a in algorithms for s in strategies]
    pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(desired_cols, names=["Algorithm", "Strategy"]))

    # summary: mean Err per (Algorithm, Strategy)
    mean_row = pivot.mean(axis=0, skipna=True)

    # mean rank (Err) computed WITHIN EACH algorithm across strategies
    mean_rank = {}
    for a in algorithms:
        err_wide = (merged[merged["Algorithm"] == a]
                    .pivot(index="dataset", columns="Strategy", values="Err")
                    .reindex(columns=strategies))
        ranks = err_wide.rank(axis=1, method="average", ascending=True)
        mean_rank[a] = ranks.mean(axis=0, skipna=True)  # Series indexed by Strategy

    # format body
    pivot_fmt = pivot.applymap(_fmt)
    pivot_fmt.index = [_latex_escape(str(x)) for x in pivot_fmt.index]

    # build LaTeX
    # 1 dataset col + vertical separators between algorithm blocks
    # format: l | c c c c c | c c c c c | c c c c c
    col_spec_parts = ["l"]
    for i, _ in enumerate(algorithms):
        col_spec_parts.append("|")
        col_spec_parts.extend(["c"] * len(strategies))
    col_spec = "".join(col_spec_parts)

    n_err_cols = len(algorithms) * len(strategies)
    last_col_idx = 1 + n_err_cols

    lines = []
    lines.append("\\begin{table}[!ht]")
    lines.append("    \\centering")
    lines.append(f"    \\begin{{tabular}}{{{col_spec}}}")
    lines.append("    \\toprule")

    # Header row 1: Algorithm group headers with vertical separation
    group_cells = ["Dataset"]
    for a in algorithms:
        group_cells.append(f"\\multicolumn{{{len(strategies)}}}{{c|}}{{{a}}}")
    group_cells[-1] = group_cells[-1].replace("|}", "}")  # remove trailing |
    lines.append("    " + " & ".join(group_cells) + " \\\\")
    lines.append("    \\midrule")


    # Header row 2: Strategy headers repeated
    strat_cells = [""]
    for a in algorithms:
        strat_cells.extend(strategies)
    lines.append("    " + " & ".join(strat_cells) + " \\\\")
    lines.append("    \\midrule")

    # Body rows
    for ds, row in pivot_fmt.iterrows():
        vals = [ds]
        for a in algorithms:
            for s in strategies:
                vals.append(row.get((a, s), fill))
        lines.append("    " + " & ".join(vals) + " \\\\")

    # Summary rows
    lines.append("    \\midrule")

    # Mean row
    mean_fmt = [ _fmt(mean_row.get((a, s))) for a in algorithms for s in strategies ]
    lines.append("    " + " & ".join(["Mean"] + mean_fmt) + " \\\\")

    # Mean rank row (per algorithm — visually separated by vertical bars)
    rank_fmt = []
    for a in algorithms:
        for s in strategies:
            rank_fmt.append(_fmt(mean_rank[a].get(s)))

    lines.append("    " + " & ".join(["Mean rank (Err)"] + rank_fmt) + " \\\\")

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")

    rr_pct = int(round(float(target_rejection_rate) * 100))
    cap = (
        f"{caption_prefix} at rejection rate {rr_pct}\\%. "
        "Mean ranks are computed separately within each algorithm."
    )
    lab = f"{label_prefix}:{rr_pct}"
    lines.append(f"    \\caption{{{cap}}}")
    lines.append(f"    \\label{{{lab}}}")
    lines.append("\\end{table}")

    return "\n".join(lines)

# ---------- Example usage ----------
if __name__ == "__main__":
    datasets_to_process = [
        "bank8FM","adult","magic","eeg","phishing", "ailerons", "deltae","cpu", 
        "puma8NH","phoneme","krkp","pc4","qsar","german","raising","diabetes",
        "liver","kc2","vertebral", "kc3","heartC","heartH","heartS","hepati",
    ]

    folder = "./exp_performance_results"
    df = load_results_df(datasets_to_process, folder)

    latex_str = make_dataset_err_table_all_algos_latex(
        df,
        target_rejection_rate=0.6,  # <-- change this
        rr_tol=1e-6,                # <-- set to 1e-3 if you have float noise
        dataset_order=datasets_to_process,
        float_fmt="{:.3f}",
        fill="."
    )

    print(latex_str)

\begin{table}[!ht]
    \centering
    \begin{tabular}{l|ccccc|ccccc|ccccc}
    \toprule
    Dataset & \multicolumn{5}{c|}{DT} & \multicolumn{5}{c|}{RF} & \multicolumn{5}{c}{XGB} \\
    \midrule
     & Uncal & Iso & Platt & RegVA & AvgVA & Uncal & Iso & Platt & RegVA & AvgVA & Uncal & Iso & Platt & RegVA & AvgVA \\
    \midrule
    bank8FM & .005 & .001 & .021 & .001 & .001 & .016 & .002 & .011 & .001 & .001 & .000 & .002 & .031 & .002 & .002 \\
    adult & .004 & .001 & .031 & .002 & .000 & .007 & .001 & .030 & .001 & .000 & .001 & .001 & .034 & .002 & .001 \\
    magic & .021 & .003 & .021 & .004 & .001 & .004 & .002 & .016 & .002 & .000 & .012 & .003 & .037 & .004 & .000 \\
    eeg & .033 & .004 & .011 & .004 & .002 & .066 & .001 & .005 & .003 & .001 & .005 & .002 & .020 & .003 & .001 \\
    phishing & .002 & .001 & .023 & .001 & .000 & .001 & .001 & .007 & .001 & .000 & .000 & .001 & .010 & .002 & .001 \\
    ailerons & .010 & .002 & .032 & .002 & .001 & .022 & .002 & .015 & .003 & 

/tmp/ipykernel_220981/3372825352.py:179: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_fmt = pivot.applymap(_fmt)


### Worst-case performance estimation - Per-dataset estimation errors

In [ ]:
import os
import glob
import pandas as pd

# ---------- Config / mappings ----------
# your lower-bound file naming: XXX_<dataset>_<strategy>_<algorithm>.csv
METHODS = {
    "naive":    ("Uncal", "gray",       "o"),
    "isotonic": ("Iso",   "darkgreen",  "^"),
    "sigmoid":  ("Platt", "darkorange", "s"),
    "regva":       ("RegVA", "darkred",    "D"),
    "avgva":      ("AvgVA", "darkred",    "P"),
}
STRATEGY_LABELS = {k: v[0] for k, v in METHODS.items()}
STRATEGY_ORDER  = [METHODS[k][0] for k in METHODS]          # ["Uncal","Platt","Iso","RegVA","AvgVA"]

ALGO_LABELS = {"tree": "DT", "rf": "RF", "xgb": "XGB"}
ALGO_ORDER  = ["DT", "RF", "XGB"]


# ---------- Loading (lower bound experiment) ----------
def _parse_lb_filename(path, prefix=""):
    """
    Expected: <dataset>_<strategy>_<algorithm>.csv
    Example:  adult_isotonic_tree.csv
    """
    base = os.path.basename(path)
    name = base[:-4]
    if prefix and name.startswith(prefix):
        name = name[len(prefix):]
    parts = name.split("_")
    if len(parts) < 3:
        return None
    dataset, strategy, algorithm = parts[0], parts[1].lower(), parts[2].lower()
    return dataset, strategy, algorithm

def load_lower_bound_results_df(folder_path="./lower_bound_results"):
    """
    Loads lower-bound CSVs.
    Required columns in each file:
      rejection_rate, abs_error, mean_accuracy, mean_lower_bound
    Returns a single dataframe with columns including:
      dataset, strategy, algorithm, rejection_rate, abs_error, ...
    """
    files = sorted(glob.glob(os.path.join(folder_path, f"{prefix}*.csv")))
    if not files:
        raise FileNotFoundError(f"No files found in {folder_path} with prefix {prefix}")

    dfs, skipped = [], []
    for fp in files:
        parsed = _parse_lb_filename(fp, prefix=prefix)
        if not parsed:
            skipped.append((fp, "filename pattern mismatch"))
            continue

        dataset, strategy, algorithm = parsed

        if strategy not in STRATEGY_LABELS:
            skipped.append((fp, f"unknown strategy: {strategy}"))
            continue
        if algorithm not in ALGO_LABELS:
            skipped.append((fp, f"unknown algorithm: {algorithm}"))
            continue

        try:
            df = pd.read_csv(fp)
        except Exception as e:
            skipped.append((fp, f"read error: {e}"))
            continue

        required = {"rejection_rate", "abs_error"}
        if not required.issubset(df.columns):
            skipped.append((fp, f"missing columns; found {list(df.columns)}"))
            continue

        df = df.copy()
        df["dataset"] = dataset
        df["strategy"] = strategy
        df["algorithm"] = algorithm

        # numeric coercion
        for c in ["rejection_rate", "coverage_rate", "abs_error", "signed_error",
                  "mean_accuracy", "mean_lower_bound"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

        dfs.append(df)

    if not dfs:
        msg = "No valid lower-bound files loaded."
        if skipped:
            msg += " Skipped: " + "; ".join([f"{os.path.basename(f)} → {r}" for f, r in skipped[:30]])
            if len(skipped) > 30:
                msg += f" ... (+{len(skipped)-30} more)"
        raise RuntimeError(msg)

    out = pd.concat(dfs, ignore_index=True)
    return out


# ---------- LaTeX helpers ----------
def _strip_leading_zero_str(s: str) -> str:
    if s.startswith("0."):
        return s[1:]
    if s.startswith("-0."):
        return "-" + s[2:]
    return s

def _latex_escape(s: str) -> str:
    return (s.replace("\\", "\\textbackslash{}")
             .replace("_", "\\_")
             .replace("%", "\\%")
             .replace("&", "\\&")
             .replace("#", "\\#")
             .replace("{", "\\{")
             .replace("}", "\\}")
             .replace("$", "\\$")
             .replace("~", "\\textasciitilde{}")
             .replace("^", "\\textasciicircum{}"))


# ---------- Main: absolute estimation error table (lower bound) ----------
def make_dataset_abs_err_table_all_algos_latex_lower_bound(
    results_df: pd.DataFrame,
    target_rejection_rate: float,
    rr_tol: float = 1e-6,
    algorithms=ALGO_ORDER,
    strategies=STRATEGY_ORDER,
    dataset_order=None,  # list of dataset names (optional)
    caption_prefix="Absolute estimation error (lower bound)",
    label_prefix="tab:lb_abs_err_all_algos",
    float_fmt="{:.3f}",
    fill="."
) -> str:
    """
    Single LaTeX table (all algorithms together) for the LOWER-BOUND experiment.

    Rows: datasets
    Cols: for each Algorithm -> for each Strategy -> abs_error (already in files)
    Plus:
      - Mean: mean abs_error across datasets per (Algorithm, Strategy)
      - Mean rank (Err): within each algorithm, rank strategies per dataset by abs_error (lower better),
                         then average across datasets.

    Selection: rejection_rate ~= target_rejection_rate within rr_tol.
    Aggregation: mean abs_error across repeats/rows for that rejection rate (if duplicates exist).
    """

    df = results_df.copy()
    df["rejection_rate"] = df["rejection_rate"].astype(float)

    sel = df[(df["rejection_rate"] - float(target_rejection_rate)).abs() <= float(rr_tol)].copy()
    if sel.empty:
        avail = sorted(df["rejection_rate"].round(6).dropna().unique())
        raise RuntimeError(
            f"No rows found for target_rejection_rate={target_rejection_rate} within rr_tol={rr_tol}. "
            f"Available rejection_rate values (sample): {avail[:20]}{' ...' if len(avail) > 20 else ''}"
        )

    sel["Algorithm"] = sel["algorithm"].map(ALGO_LABELS)
    sel["Strategy"]  = sel["strategy"].map(STRATEGY_LABELS)
    sel = sel.dropna(subset=["Algorithm", "Strategy"])

    # mean abs_error per dataset x algorithm x strategy
    merged = (
        sel.groupby(["dataset", "Algorithm", "Strategy"], as_index=False)["abs_error"]
           .mean()
           .rename(columns={"abs_error": "Err"})
    )

    # Pivot: rows=datasets, cols=(Algorithm, Strategy), values=Err
    pivot = merged.pivot(index="dataset", columns=["Algorithm", "Strategy"], values="Err")

    if dataset_order is not None:
        pivot = pivot.reindex(dataset_order)

    # enforce column order
    desired_cols = [(a, s) for a in algorithms for s in strategies]
    pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(desired_cols, names=["Algorithm", "Strategy"]))

    # summary: mean Err per (Algorithm, Strategy)
    mean_row = pivot.mean(axis=0, skipna=True)

    # mean rank within each algorithm
    mean_rank = {}
    for a in algorithms:
        err_wide = (
            merged[merged["Algorithm"] == a]
            .pivot(index="dataset", columns="Strategy", values="Err")
            .reindex(columns=strategies)
        )
        ranks = err_wide.rank(axis=1, method="average", ascending=True)
        mean_rank[a] = ranks.mean(axis=0, skipna=True)

    def _fmt(x):
        if pd.isnull(x):
            return fill
        return _strip_leading_zero_str(float_fmt.format(float(x)))

    pivot_fmt = pivot.map(_fmt)
    pivot_fmt.index = [_latex_escape(str(x)) for x in pivot_fmt.index]

    # column spec: l | (c...)*algos with separators
    col_spec_parts = ["l"]
    for _ in algorithms:
        col_spec_parts.append("|")
        col_spec_parts.extend(["c"] * len(strategies))
    col_spec = "".join(col_spec_parts)

    lines = []
    lines.append("\\begin{landscape}")
    lines.append("\\begin{table}[!ht]")
    lines.append("    \\centering")
    lines.append("    \\footnotesize")
    lines.append(f"    \\begin{{tabular}}{{{col_spec}}}")
    lines.append("    \\toprule")

    # Header row 1: algorithm blocks
    group_cells = ["Dataset"]
    for a in algorithms:
        group_cells.append(f"\\multicolumn{{{len(strategies)}}}{{c|}}{{{a}}}")
    group_cells[-1] = group_cells[-1].replace("|}", "}")
    lines.append("    " + " & ".join(group_cells) + " \\\\")
    lines.append("    \\midrule")

    # Header row 2: strategies repeated
    strat_cells = [""]
    for _a in algorithms:
        strat_cells.extend(strategies)
    lines.append("    " + " & ".join(strat_cells) + " \\\\")
    lines.append("    \\midrule")

    # Body rows
    for ds, row in pivot_fmt.iterrows():
        vals = [ds]
        for a in algorithms:
            for s in strategies:
                vals.append(row.get((a, s), fill))
        lines.append("    " + " & ".join(vals) + " \\\\")

    # Summary
    lines.append("    \\midrule")
    mean_fmt = [_fmt(mean_row.get((a, s))) for a in algorithms for s in strategies]
    lines.append("    " + " & ".join(["Mean"] + mean_fmt) + " \\\\")

    rank_fmt = []
    for a in algorithms:
        for s in strategies:
            rank_fmt.append(_fmt(mean_rank[a].get(s)))
    lines.append("    " + " & ".join(["Mean rank"] + rank_fmt) + " \\\\")

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")

    rr_pct = int(round(float(target_rejection_rate) * 100))
    cap = (
        f"Worst-case performance estimation - Absolute errors at rejection rate {rr_pct}\\%. "
        "Mean ranks are computed separately within each algorithm."
    )
    lab = f"{label_prefix}:{rr_pct}"
    lines.append(f"    \\caption{{{cap}}}")
    lines.append(f"    \\label{{{lab}}}")
    lines.append("\\end{table}")
    lines.append("\\end{landscape}")
    
    return "\n".join(lines)


# ---------- Example usage ----------
if __name__ == "__main__":
    datasets_to_process = [
        "bank8FM","adult","magic","eeg","phishing", "ailerons", "deltae","cpu",
        "puma8NH","phoneme","krkp","pc4","qsar","german","raising","diabetes",
        "liver","kc2","vertebral", "kc3","heartC","heartH","heartS","hepati",
    ]

    folder = "./lower_bound_results"
    df_lb = load_lower_bound_results_df(folder_path=folder, prefix="XXX_")

    latex_str = make_dataset_abs_err_table_all_algos_latex_lower_bound(
        df_lb,
        target_rejection_rate=0.9,
        rr_tol=1e-6,
        dataset_order=datasets_to_process,
        float_fmt="{:.3f}",
        fill="."
    )

    print(latex_str)

\begin{landscape}
\begin{table}[!ht]
    \centering
    \footnotesize
    \begin{tabular}{l|ccccc|ccccc|ccccc}
    \toprule
    Dataset & \multicolumn{5}{c|}{DT} & \multicolumn{5}{c|}{RF} & \multicolumn{5}{c}{XGB} \\
    \midrule
     & Uncal & Iso & Platt & RegVA & AvgVA & Uncal & Iso & Platt & RegVA & AvgVA & Uncal & Iso & Platt & RegVA & AvgVA \\
    \midrule
    bank8FM & .011 & .004 & .025 & .000 & .002 & .014 & .004 & .018 & .004 & .003 & .000 & .003 & .049 & .007 & .004 \\
    adult & .007 & .001 & .040 & .000 & .000 & .008 & .000 & .041 & .000 & .000 & .002 & .002 & .047 & .002 & .001 \\
    magic & .031 & .005 & .040 & .002 & .004 & .005 & .004 & .029 & .003 & .001 & .008 & .007 & .057 & .004 & .000 \\
    eeg & .058 & .011 & .059 & .001 & .007 & .051 & .003 & .007 & .005 & .003 & .000 & .004 & .027 & .003 & .000 \\
    phishing & .005 & .000 & .023 & .003 & .001 & .002 & .004 & .009 & .001 & .001 & .000 & .002 & .020 & .004 & .002 \\
    ailerons & .018 & .005 & .039 & .000 &

### Expected performance estimation - AvgVA interval table (per dataset)

In [29]:
import os
import re
import glob
import pandas as pd

# ---------- Config ----------
ALGO_LABELS = {"tree": "Decision tree", "rf": "Random forest", "xgb": "Xgboost"}
ALGO_ORDER = ["Decision tree", "Random forest", "Xgboost"]
ROW_END = r" \\\\"  # LaTeX row terminator

# Expected pattern: <dataset>_avgva_<algorithm>.csv
FNAME_RE = re.compile(
    r"^(?P<dataset>[^_]+)_avgva_(?P<algorithm>[^_]+)\.csv$",
    re.IGNORECASE,
)


def _parse_avgva_filename(path):
    base = os.path.basename(path)
    m = FNAME_RE.match(base)
    if not m:
        return None
    d = m.groupdict()
    return d["dataset"], d["algorithm"].lower()


def _strip_leading_zero_str(s: str) -> str:
    if s.startswith("0."):
        return s[1:]
    if s.startswith("-0."):
        return "-" + s[2:]
    return s


def _latex_escape(s: str) -> str:
    return (
        s.replace("\\", "\\textbackslash{}")
        .replace("_", "\\_")
        .replace("%", "\\%")
        .replace("&", "\\&")
        .replace("#", "\\#")
        .replace("{", "\\{")
        .replace("}", "\\}")
        .replace("$", "\\$")
        .replace("~", "\\textasciitilde{}")
        .replace("^", "\\textasciicircum{}")
    )


def load_avgva_interval_df(folder_path="./exp_performance_results"):
    """Load only AvgVA files and compute interval size (upper-lower)."""
    files = sorted(glob.glob(os.path.join(folder_path, "*avgva*.csv")))
    if not files:
        raise FileNotFoundError(f"No '*avgva*.csv' files found in {folder_path}")

    dfs, skipped = [], []
    for fp in files:
        parsed = _parse_avgva_filename(fp)
        if not parsed:
            skipped.append((fp, "filename pattern mismatch"))
            continue

        dataset, algorithm = parsed
        if algorithm not in ALGO_LABELS:
            skipped.append((fp, f"unknown algorithm: {algorithm}"))
            continue

        try:
            df = pd.read_csv(fp)
        except Exception as e:
            skipped.append((fp, f"read error: {e}"))
            continue

        required = {"rejection_rate", "lower_bound", "upper_bound", "accuracy"}
        if not required.issubset(df.columns):
            skipped.append((fp, f"missing columns; found {list(df.columns)}"))
            continue

        df = df.copy()
        for c in ["rejection_rate", "lower_bound", "upper_bound", "accuracy"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df["size"] = df["upper_bound"] - df["lower_bound"]
        df["dataset"] = dataset
        df["algorithm"] = algorithm
        dfs.append(df[["dataset", "algorithm", "rejection_rate", "lower_bound", "upper_bound", "size", "accuracy"]])

    if not dfs:
        msg = "No valid AvgVA files loaded."
        if skipped:
            msg += " Skipped: " + "; ".join([f"{os.path.basename(f)} -> {r}" for f, r in skipped[:20]])
        raise RuntimeError(msg)

    out = pd.concat(dfs, ignore_index=True)
    out["Algorithm"] = out["algorithm"].map(ALGO_LABELS)
    out = out.dropna(subset=["Algorithm"])
    return out


def make_avgva_interval_table_latex(
    results_df: pd.DataFrame,
    target_rejection_rate: float,
    rr_tol: float = 1e-6,
    algorithms=ALGO_ORDER,
    dataset_order=None,
    caption_prefix="Per-dataset AvgVA interval summary ($C_0$, $C_1$, Size, and observed accuracy)",
    label_prefix="tab:avgva_interval",
    float_fmt="{:.3f}",
    fill=".",
):
    """
    One LaTeX table, rows=datasets.
    For each algorithm: LB, UB, Size (= UB-LB), Acc.
    Underlines Acc when it falls outside [LB, UB].
    """
    df = results_df.copy()
    df["rejection_rate"] = pd.to_numeric(df["rejection_rate"], errors="coerce")

    sel = df[(df["rejection_rate"] - float(target_rejection_rate)).abs() <= float(rr_tol)].copy()
    if sel.empty:
        avail = sorted(df["rejection_rate"].dropna().round(6).unique())
        raise RuntimeError(
            f"No rows found for target_rejection_rate={target_rejection_rate} within rr_tol={rr_tol}. "
            f"Available values (sample): {avail[:20]}{' ...' if len(avail) > 20 else ''}"
        )

    agg = (
        sel.groupby(["dataset", "Algorithm"], as_index=False)
        .agg(
            lower_bound=("lower_bound", "mean"),
            upper_bound=("upper_bound", "mean"),
            size=("size", "mean"),
            accuracy=("accuracy", "mean"),
        )
    )

    metrics = ["lower_bound", "upper_bound", "size", "accuracy"]
    metric_labels = {
        "lower_bound": "$C_0$",
        "upper_bound": "$C_1$",
        "size": "Size",
        "accuracy": "Acc.",
    }

    pivot = agg.pivot(index="dataset", columns="Algorithm", values=metrics)
    pivot = pivot.swaplevel(0, 1, axis=1)

    desired_cols = pd.MultiIndex.from_tuples(
        [(a, m) for a in algorithms for m in metrics],
        names=["Algorithm", "Metric"],
    )
    pivot = pivot.reindex(columns=desired_cols)

    if dataset_order is not None:
        pivot = pivot.reindex(dataset_order)
    else:
        pivot = pivot.sort_index()

    def _fmt(x):
        if pd.isnull(x):
            return fill
        return _strip_leading_zero_str(float_fmt.format(float(x)))

    def _acc_outside_interval(acc, lb, ub):
        if pd.isnull(acc) or pd.isnull(lb) or pd.isnull(ub):
            return False
        return (acc < lb) or (acc > ub)

    col_means = pivot.mean(axis=0, skipna=True)

    # Dataset + 4 columns per algorithm
    col_spec_parts = ["l"]
    for _ in algorithms:
        col_spec_parts.append("|")
        col_spec_parts.extend(["c"] * len(metrics))
    col_spec = "".join(col_spec_parts)

    lines = []
    lines.append("\\begin{table}[!ht]")
    lines.append("    \\centering")
    lines.append("    \\footnotesize")
    lines.append(f"    \\begin{{tabular}}{{{col_spec}}}")
    lines.append("    \\toprule")

    # Header row 1: algorithm blocks
    group_cells = ["Dataset"]
    for a in algorithms:
        group_cells.append(f"\\multicolumn{{{len(metrics)}}}{{c|}}{{{a}}}")
    group_cells[-1] = group_cells[-1].replace("|}", "}")
    lines.append("    " + " & ".join(group_cells) + ROW_END)
    lines.append("    \\midrule")

    # Header row 2: metric labels
    sub_cells = [""]
    for _ in algorithms:
        sub_cells.extend([metric_labels[m] for m in metrics])
    lines.append("    " + " & ".join(sub_cells) + ROW_END)
    lines.append("    \\midrule")

    # Body rows
    for ds in pivot.index:
        row = pivot.loc[ds]
        vals = [_latex_escape(str(ds))]

        for a in algorithms:
            lb = row.get((a, "lower_bound"))
            ub = row.get((a, "upper_bound"))
            sz = row.get((a, "size"))
            acc = row.get((a, "accuracy"))

            vals.append(_fmt(lb))
            vals.append(_fmt(ub))
            vals.append(_fmt(sz))

            acc_txt = _fmt(acc)
            if _acc_outside_interval(acc, lb, ub):
                acc_txt = f"\\underline{{{acc_txt}}}"
            vals.append(acc_txt)

        lines.append("    " + " & ".join(vals) + ROW_END)

    # Mean row
    lines.append("    \\midrule")
    mean_vals = [_fmt(col_means.get((a, m))) for a in algorithms for m in metrics]
    lines.append("    " + " & ".join(["Mean"] + mean_vals) + ROW_END)

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")

    rr_pct = int(round(float(target_rejection_rate) * 100))
    lines.append(f"    \\caption{{{caption_prefix} at rejection rate {rr_pct}\\%.}}")
    lines.append(f"    \\label{{{label_prefix}:{rr_pct}}}")
    lines.append("\\end{table}")

    return "\n".join(lines)


# ---------- Example usage ----------
if __name__ == "__main__":
    datasets_to_process = [
        "bank8FM", "adult", "magic", "eeg", "phishing", "ailerons", "deltae", "cpu",
        "puma8NH", "phoneme", "krkp", "pc4", "qsar", "german", "raising", "diabetes",
        "liver", "kc2", "vertebral", "kc3", "heartC", "heartH", "heartS", "hepati",
    ]

    df_avgva = load_avgva_interval_df(folder_path="./exp_performance_results")
    df_avgva = df_avgva[df_avgva["dataset"].isin(datasets_to_process)].copy()

    latex_str = make_avgva_interval_table_latex(
        df_avgva,
        target_rejection_rate=0.1,
        rr_tol=1e-6,
        dataset_order=datasets_to_process,
        float_fmt="{:.3f}",
        fill=".",
    )

    print(latex_str)

\begin{table}[!ht]
    \centering
    \footnotesize
    \begin{tabular}{l|cccc|cccc|cccc}
    \toprule
    Dataset & \multicolumn{4}{c|}{Decision tree} & \multicolumn{4}{c|}{Random forest} & \multicolumn{4}{c}{Xgboost} \\\\
    \midrule
     & $C_0$ & $C_1$ & Size & Acc. & $C_0$ & $C_1$ & Size & Acc. & $C_0$ & $C_1$ & Size & Acc. \\\\
    \midrule
    bank8FM & .941 & .958 & .017 & .950 & .961 & .971 & .010 & .969 & .963 & .975 & .012 & .971 \\\\
    adult & .876 & .886 & .010 & .882 & .886 & .889 & .003 & .887 & .898 & .903 & .004 & .900 \\\\
    magic & .850 & .866 & .015 & .859 & .901 & .907 & .006 & .905 & .901 & .909 & .008 & .907 \\\\
    eeg & .749 & .768 & .019 & .757 & .940 & .948 & .007 & .945 & .946 & .955 & .009 & .952 \\\\
    phishing & .943 & .960 & .017 & .952 & .986 & .991 & .005 & .990 & .987 & .993 & .007 & .992 \\\\
    ailerons & .871 & .889 & .019 & .880 & .894 & .903 & .008 & .899 & .905 & .916 & .011 & .911 \\\\
    deltae & .888 & .908 & .021 & .900 & .902 & .9

### Expected performance estimation - AvgVA interval table by rejection rate (dataset-averaged)

In [30]:
import os
import re
import glob
import pandas as pd

# ---------- Config ----------
ALGO_LABELS = {"tree": "Decision tree", "rf": "Random forest", "xgb": "XGBoost"}
ALGO_ORDER = ["Decision tree", "Random forest", "XGBoost"]
ROW_END = r" \\\\"  # LaTeX row terminator

# Expected pattern: <dataset>_avgva_<algorithm>.csv
FNAME_RE = re.compile(
    r"^(?P<dataset>[^_]+)_avgva_(?P<algorithm>[^_]+)\.csv$",
    re.IGNORECASE,
)


def _parse_avgva_filename(path):
    base = os.path.basename(path)
    m = FNAME_RE.match(base)
    if not m:
        return None
    d = m.groupdict()
    return d["dataset"], d["algorithm"].lower()


def _strip_leading_zero_str(s: str) -> str:
    if s.startswith("0."):
        return s[1:]
    if s.startswith("-0."):
        return "-" + s[2:]
    return s


def load_avgva_interval_df(folder_path="./exp_performance_results"):
    files = sorted(glob.glob(os.path.join(folder_path, "*avgva*.csv")))
    if not files:
        raise FileNotFoundError(f"No '*avgva*.csv' files found in {folder_path}")

    dfs, skipped = [], []
    for fp in files:
        parsed = _parse_avgva_filename(fp)
        if not parsed:
            skipped.append((fp, "filename pattern mismatch"))
            continue

        dataset, algorithm = parsed
        if algorithm not in ALGO_LABELS:
            skipped.append((fp, f"unknown algorithm: {algorithm}"))
            continue

        try:
            df = pd.read_csv(fp)
        except Exception as e:
            skipped.append((fp, f"read error: {e}"))
            continue

        required = {"rejection_rate", "lower_bound", "upper_bound", "accuracy"}
        if not required.issubset(df.columns):
            skipped.append((fp, f"missing columns; found {list(df.columns)}"))
            continue

        df = df.copy()
        for c in ["rejection_rate", "lower_bound", "upper_bound", "accuracy"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df["size"] = df["upper_bound"] - df["lower_bound"]
        df["dataset"] = dataset
        df["algorithm"] = algorithm
        dfs.append(df[["dataset", "algorithm", "rejection_rate", "lower_bound", "upper_bound", "size", "accuracy"]])

    if not dfs:
        msg = "No valid AvgVA files loaded."
        if skipped:
            msg += " Skipped: " + "; ".join([f"{os.path.basename(f)} -> {r}" for f, r in skipped[:20]])
        raise RuntimeError(msg)

    out = pd.concat(dfs, ignore_index=True)
    out["Algorithm"] = out["algorithm"].map(ALGO_LABELS)
    out = out.dropna(subset=["Algorithm"])
    return out


def make_avgva_interval_by_rejection_table_latex(
    results_df: pd.DataFrame,
    algorithms=ALGO_ORDER,
    caption="AvgVA interval summary by rejection rate (averaged across datasets)",
    label="tab:avgva_interval_by_rejection",
    float_fmt="{:.3f}",
    fill=".",
):
    """
    Builds one LaTeX table with rows = rejection rates and columns per algorithm:
      C0, C1, Size, Acc.
    No final mean row is included.
    """
    df = results_df.copy()
    df["rejection_rate"] = pd.to_numeric(df["rejection_rate"], errors="coerce")
    df["rejection_rate"] = df["rejection_rate"].round(1)

    # Average across datasets for each rejection rate and algorithm
    agg = (
        df.groupby(["rejection_rate", "Algorithm"], as_index=False)
        .agg(
            lower_bound=("lower_bound", "mean"),
            upper_bound=("upper_bound", "mean"),
            size=("size", "mean"),
            accuracy=("accuracy", "mean"),
        )
    )

    metrics = ["lower_bound", "upper_bound", "size", "accuracy"]
    metric_labels = {
        "lower_bound": "$C_0$",
        "upper_bound": "$C_1$",
        "size": "Size",
        "accuracy": "Acc.",
    }

    pivot = agg.pivot(index="rejection_rate", columns="Algorithm", values=metrics)
    pivot = pivot.swaplevel(0, 1, axis=1)

    desired_cols = pd.MultiIndex.from_tuples(
        [(a, m) for a in algorithms for m in metrics],
        names=["Algorithm", "Metric"],
    )
    pivot = pivot.reindex(columns=desired_cols)

    rr_order = [round(x / 10, 1) for x in range(1, 10)]
    pivot = pivot.reindex(rr_order)

    def _fmt(x):
        if pd.isnull(x):
            return fill
        return _strip_leading_zero_str(float_fmt.format(float(x)))

    col_spec_parts = ["c"]
    for _ in algorithms:
        col_spec_parts.append("|")
        col_spec_parts.extend(["c"] * len(metrics))
    col_spec = "".join(col_spec_parts)

    lines = []
    lines.append("\\begin{table}[!ht]")
    lines.append("    \\centering")
    lines.append("    \\footnotesize")
    lines.append(f"    \\begin{{tabular}}{{{col_spec}}}")
    lines.append("    \\toprule")

    # Header row 1
    group_cells = ["Rejection rate"]
    for a in algorithms:
        group_cells.append(f"\\multicolumn{{{len(metrics)}}}{{c|}}{{{a}}}")
    group_cells[-1] = group_cells[-1].replace("|}", "}")
    lines.append("    " + " & ".join(group_cells) + ROW_END)
    lines.append("    \\midrule")

    # Header row 2
    sub_cells = [""]
    for _ in algorithms:
        sub_cells.extend([metric_labels[m] for m in metrics])
    lines.append("    " + " & ".join(sub_cells) + ROW_END)
    lines.append("    \\midrule")

    # Body rows (no mean row)
    for rr in rr_order:
        row = [f"{int(round(rr * 100))}\\%"]
        for a in algorithms:
            for m in metrics:
                row.append(_fmt(pivot.loc[rr, (a, m)] if rr in pivot.index else pd.NA))
        lines.append("    " + " & ".join(row) + ROW_END)

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")
    lines.append(f"    \\caption{{{caption}.}}")
    lines.append(f"    \\label{{{label}}}")
    lines.append("\\end{table}")

    return "\n".join(lines)


# ---------- Example usage ----------
if __name__ == "__main__":
    datasets_to_process = [
        "bank8FM", "adult", "magic", "eeg", "phishing", "ailerons", "deltae", "cpu",
        "puma8NH", "phoneme", "krkp", "pc4", "qsar", "german", "raising", "diabetes",
        "liver", "kc2", "vertebral", "kc3", "heartC", "heartH", "heartS", "hepati",
    ]

    df_avgva = load_avgva_interval_df(folder_path="./exp_performance_results")
    df_avgva = df_avgva[df_avgva["dataset"].isin(datasets_to_process)].copy()

    latex_str = make_avgva_interval_by_rejection_table_latex(
        df_avgva,
        float_fmt="{:.3f}",
        fill=".",
    )

    print(latex_str)

\begin{table}[!ht]
    \centering
    \footnotesize
    \begin{tabular}{c|cccc|cccc|cccc}
    \toprule
    Rejection rate & \multicolumn{4}{c|}{Decision tree} & \multicolumn{4}{c|}{Random forest} & \multicolumn{4}{c}{XGBoost} \\\\
    \midrule
     & $C_0$ & $C_1$ & Size & Acc. & $C_0$ & $C_1$ & Size & Acc. & $C_0$ & $C_1$ & Size & Acc. \\\\
    \midrule
    10\% & .810 & .865 & .055 & .837 & .851 & .900 & .048 & .877 & .846 & .896 & .051 & .870 \\\\
    20\% & .837 & .886 & .049 & .861 & .874 & .920 & .046 & .900 & .869 & .918 & .049 & .894 \\\\
    30\% & .861 & .906 & .045 & .882 & .896 & .939 & .044 & .921 & .889 & .935 & .046 & .912 \\\\
    40\% & .882 & .922 & .039 & .901 & .911 & .953 & .041 & .936 & .905 & .949 & .044 & .928 \\\\
    50\% & .901 & .935 & .034 & .918 & .927 & .966 & .039 & .950 & .918 & .960 & .041 & .942 \\\\
    60\% & .918 & .947 & .028 & .931 & .940 & .975 & .035 & .960 & .931 & .970 & .039 & .952 \\\\
    70\% & .932 & .955 & .024 & .943 & .951 & .983 & .0